# 🧪 Sesión 32 · Despliegue en la Nube (Render / Codespaces) y Env Vars

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Dónde guardas la contraseña de la base de datos?

---

## 🖥️ Así se verá el resultado final

```text
$ DATABASE_URL=sqlite:///biblioteca.db SECRET_KEY=clave uvicorn app:app
GET /api/config
HTTP/1.1 200 OK
{"entorno":"dev","debug":false}

$ uvicorn app:app   (sin SECRET_KEY)
RuntimeError: Falta la variable SECRET_KEY
[el servidor no arranca]

La respuesta de /api/config NUNCA incluye SECRET_KEY.
```

_Si el servidor arranca sin secreto, el error aparecerá en producción en el peor momento._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Leer configuración desde variables de entorno.
- Fallar rápido si falta un secreto crítico.
- Verificar que los secretos nunca se exponen en la respuesta.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] La configuración no secreta se expone → HTTP 200
- [ ] La configuración obligatoria falla rápido
- [ ] Variables documentadas
- [ ] .env fuera del repositorio

## 🧩 El reto

Implementa una configuración centralizada que lea variables de entorno, valide las obligatorias y no exponga secretos.

**Requisitos del contrato**

- config.py lee DATABASE_URL y SECRET_KEY como obligatorias.
- ENTORNO tiene valor por defecto dev y DEBUG por defecto false.
- Si falta una obligatoria, el arranque falla con un error claro.
- GET /api/config devuelve solo configuración no sensible.
- .env.example documenta las variables y .gitignore excluye .env.

> **Pista conceptual:** La configuración se lee una vez al arrancar. Si el código de negocio llama `os.getenv` por todas partes, no hay un único contrato de configuración.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion32.md` con la lista de variables y su origen.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [ ]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `config.py`, `.env.example`, `.gitignore`, `app.py`


### 📄 `config.py`

Módulo de configuración.

In [ ]:
%%writefile config.py
import os

def cargar_config():
    # Lee, valida y devuelve la configuración.
    pass


### 📄 `.env.example`

Plantilla de variables.

In [ ]:
%%writefile .env.example
# Documenta las variables necesarias, sin valores reales.


### 📄 `.gitignore`

Evita subir secretos.

In [ ]:
%%writefile .gitignore
.env


### 📄 `app.py`

API que usa la configuración.

In [ ]:
%%writefile app.py
from fastapi import FastAPI
from config import cargar_config

app = FastAPI(title="Configuración segura")

# Expón GET /api/config sin secretos.


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [ ]:
from verificar import probar_http

casos = [{'descripcion': 'La configuración no secreta se expone',
  'metodo': 'GET',
  'ruta': '/api/config',
  'estado': 200,
  'json_contiene': {'entorno': 'dev', 'debug': False},
  'no_contiene': ['SECRET_KEY', 'clave-de-prueba']}]

probar_http(casos, archivo="app.py", limpiar=[], env={'DATABASE_URL': 'sqlite:///biblioteca.db', 'SECRET_KEY': 'clave-de-prueba', 'ENTORNO': 'dev'})


In [ ]:
from verificar import revisar_python, resumen

resultados = []
resultados.append(revisar_python("import importlib, os, sys\nfor clave in (\"DATABASE_URL\", \"SECRET_KEY\"):\n    os.environ.pop(clave, None)\nos.environ[\"ENTORNO\"] = \"dev\"\nsys.modules.pop(\"config\", None)\nconfig = importlib.import_module(\"config\")\ntry:\n    config.cargar_config()\nexcept RuntimeError:\n    pass\nelse:\n    raise AssertionError(\"cargar_config() debería fallar si faltan variables obligatorias\")\n", "La configuración obligatoria falla rápido"))
resumen(resultados)


In [ ]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo(".env.example", regex=["DATABASE_URL", "SECRET_KEY"], descripcion="Variables documentadas"))
resultados.append(revisar_archivo(".gitignore", exigir=[".env"], descripcion=".env fuera del repositorio"))
resumen(resultados)


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion32.md` a `auditorias/auditoria_sesion32.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
